In [1]:
from importlib import reload
import TRECCASTeval as trec
import OpenSearchSimpleAPI as osearch

osearch = reload(osearch)

test_bed = trec.ConvSearchEvaluation()
opensearch = osearch.OSsimpleAPI()

In [2]:
conv_topic_idx = 0

test_bed.train_topics

[{'number': 1,
  'description': "Considering career options for becoming a physician's assistant vs a nurse.  Discussion topics include required education (including time, cost), salaries, and which is better overall.",
  'turn': [{'number': 1, 'raw_utterance': "What is a physician's assistant?"},
   {'number': 2,
    'raw_utterance': 'What are the educational requirements required to become one?'},
   {'number': 3, 'raw_utterance': 'What does it cost?'},
   {'number': 4,
    'raw_utterance': "What's the average starting salary in the UK?"},
   {'number': 5, 'raw_utterance': 'What about in the US?'},
   {'number': 6,
    'raw_utterance': 'What school subjects are needed to become a registered nurse?'},
   {'number': 7, 'raw_utterance': 'What is the PA average salary vs an RN?'},
   {'number': 8,
    'raw_utterance': 'What the difference between a PA and a nurse practitioner?'},
   {'number': 9, 'raw_utterance': 'Do NPs or PAs make more?'},
   {'number': 10, 'raw_utterance': 'Is a PA ab

In [3]:
test_bed.test_relevance_judgments

,topic_turn_id,dummy,docid,rel
0,31_1,Q0,CAR_116d829c4c800c2fc70f11692fec5e8c7e975250,0
1,31_1,Q0,CAR_1463f964653c5c9f614a0a88d26b175e4a8120f1,1
2,31_1,Q0,CAR_172e16e89ea3d5546e53384a27c3be299bcfe968,2
3,31_1,Q0,CAR_1c93ef499a0c2856c4a857b0cb4720c380dda476,0
4,31_1,Q0,CAR_2174ad0aa50712ff24035c23f59a3c2b43267650,3
...,...,...,...,...
29345,79_9,Q0,MARCO_8795229,0
29346,79_9,Q0,MARCO_8795231,0
29347,79_9,Q0,MARCO_8795233,0
29348,79_9,Q0,MARCO_8795236,0


In [4]:
from opensearchpy import NotFoundError


def get_relevances_judgement_of_turn(relevance_judgments, turn_id):
    filter = relevance_judgments['topic_turn_id'].map(lambda x: x == turn_id)
    return relevance_judgments[filter]

def get_rel_and_doc_body(relevance_judgments, turn_id):
    relevances = get_relevances_judgement_of_turn(relevance_judgments,turn_id)
    for row in relevances.values:
        doc_id = row[2]
        try:
            doc_body = opensearch.get_doc_body(doc_id)
        except NotFoundError:
            return 
        rel = row[3]/4
        return doc_body,rel

#get_rel_and_doc_body(test_bed.train_relevance_judgments,'1_6')


In [5]:
triplets = []

for topic in test_bed.train_topics:
    conv_id = topic['number']

    if conv_id not in (1, 2, 4, 7, 15, 17,18,22,23,24,25,27,30):
        continue

    print()
    print(conv_id, "  ", topic['title'])

    for turn in topic['turn']:
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)
        
        print(topic_turn_id, utterance)
        rel_and_doc = get_rel_and_doc_body(test_bed.train_relevance_judgments, topic_turn_id)
        if(rel_and_doc == None):
            continue
        triplet = [utterance,rel_and_doc[0], rel_and_doc[1]]
        triplets.append(triplet)
        #triplet = [utterance]
        print("TRIPLET:" + str(triplet))


1    Career choice for Nursing and Physician's Assistant
1_1 What is a physician's assistant?
TRIPLET:["What is a physician's assistant?", 'Physician assistants work under the supervision of a physician or surgeon; however, their specific duties and the extent to which they must be supervised differ from state to state. Physician assistants work in all areas of medicine, including primary care and family medicine, emergency medicine, and psychiatry.The work of physician assistants depends in large part on their specialty and what their supervising physician needs them to do.For example, a physician assistant working in surgery may close incisions and provide care before and after the operation.he work of physician assistants depends in large part on their specialty and what their supervising physician needs them to do. For example, a physician assistant working in surgery may close incisions and provide care before and after the operation.', 0.5]
1_2 What are the educational requireme

In [6]:
triplet

['Tell me about how I can share files.',
 "Step 1: Find the file you want to share. Open up the File Explorer and navigate to the file or files you want to share. You can share any file type, although you cannot share any file type with any app. (For example, you can't share DLL files to Facebook -- I tried). Select the file you want to share by clicking on it.tep 1: Find the file you want to share. Open up the File Explorer and navigate to the file or files you want to share. You can share any file type, although you cannot share any file type with any app. (For example, you can't share DLL files to Facebook -- I tried). Select the file you want to share by clicking on it.",
 0.5]

In [7]:
# Function to convert relevance score to binary label
# def convert_to_binary_label(relevance_score):
#     """Converts relevance score to binary label (0 or 1)."""
#     return 1 if relevance_score > 0 else 0

In [8]:
# #Convert relevance scores to binary labels (0 or 1)
# for i in range(len(triplets)):
#     triplet = triplets[i]
#     relevance_score = triplet[2]  # Relevance score is at index 2

#     # Convert relevance score to binary label
#     binary_label = convert_to_binary_label(relevance_score)

#     # Replace the relevance score with the binary label
#     triplet[2] = binary_label

#     # Update the triplet in the list
#     triplets[i] = triplet

#     print(f"Updated TRIPLET: {triplet}")

Updated TRIPLET: ["What is a physician's assistant?", 'Physician assistants work under the supervision of a physician or surgeon; however, their specific duties and the extent to which they must be supervised differ from state to state. Physician assistants work in all areas of medicine, including primary care and family medicine, emergency medicine, and psychiatry.The work of physician assistants depends in large part on their specialty and what their supervising physician needs them to do.For example, a physician assistant working in surgery may close incisions and provide care before and after the operation.he work of physician assistants depends in large part on their specialty and what their supervising physician needs them to do. For example, a physician assistant working in surgery may close incisions and provide care before and after the operation.', 1]
Updated TRIPLET: ['What are the educational requirements required to become one?', 'Video: Physician Assistant: Educational Re

In [9]:
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error



In [10]:
# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [11]:
# Function to extract CLS embeddings
def extract_cls(query_pairs, embeddings, batch_size=32):
    for batch_idx in range(0, len(query_pairs), batch_size):
        batch_data = query_pairs[batch_idx:batch_idx + batch_size]
        inputs = tokenizer.batch_encode_plus(batch_data, 
                                             return_tensors='pt', 
                                             add_special_tokens=True,  
                                             max_length=512,
                                             truncation=True,
                                             padding=True)
        with torch.no_grad():
            inputs.to(device)
            outputs = model(**inputs)
        
        batch_cls = outputs['last_hidden_state'][:, 0, :]  # Extract CLS embeddings
        batch_cls = torch.nn.functional.normalize(batch_cls, p=2, dim=1)  # L2 normalize
        
        embeddings[batch_idx:batch_idx + batch_size] = batch_cls.cpu()  # Store embeddings
    return embeddings

In [12]:
# Prepare embeddings and relevance labels
embeddings = np.zeros((len(triplets), 768))  # 768 is the size of BERT's CLS vector
relevance_labels = [pair[2] for pair in triplets]  # Extract relevance scores

# Extract CLS embeddings for each query-passage pair
embeddings = extract_cls([(pair[0], pair[1]) for pair in triplets], embeddings=embeddings, batch_size=32)

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
c:\Users\Goncalo\anaconda3\envs\nlp-cv-ir\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


In [15]:
# Train a classifier using the embeddings and relevance labels
X_train, X_test, y_train, y_test = train_test_split(embeddings, relevance_labels, test_size=0.2, random_state=42)


In [16]:

# Train the classifier (now using regression instead of classification)
regressor = LinearRegression()
regressor.fit(X_train, y_train)


LinearRegression()

In [17]:
# Predict relevance scores on the test set
y_pred = regressor.predict(X_test)


In [18]:
# Evaluate the model (using mean squared error)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 0.44844305659709743


In [34]:
top1000passages = opensearch.search_QSL(numDocs=1000)
print(top1000passages)

    _index _type            _id  _score  \
0     kwiz  _doc  MARCO_1000486     1.0   
1     kwiz  _doc  MARCO_1001206     1.0   
2     kwiz  _doc  MARCO_1001213     1.0   
3     kwiz  _doc  MARCO_1001214     1.0   
4     kwiz  _doc  MARCO_1001929     1.0   
..     ...   ...            ...     ...   
995   kwiz  _doc  MARCO_1632364     1.0   
996   kwiz  _doc  MARCO_1632365     1.0   
997   kwiz  _doc  MARCO_1632366     1.0   
998   kwiz  _doc  MARCO_1633800     1.0   
999   kwiz  _doc  MARCO_1633808     1.0   

                                      _source.contents    _source.doc  \
0    The following medical conditions are some of t...  MARCO_1000486   
1    What is the difference between observation and...  MARCO_1001206   
2    The difference between an observational study ...  MARCO_1001213   
3    A natural experiment is an empirical study in ...  MARCO_1001214   
4    In older children and adults, forceful hyperex...  MARCO_1001929   
..                                           

In [26]:
queries = ["What is a physician's assistant?"]
#topp1000_passage = [opensearch.search_body(query, numDocs=1000)["_id"] for query in queries]

#query_passage_pairs = [(query, passage) for query in queries for passage in topp1000_passage]
#query_passage_pairs

query_passage_pairs = []

for query in queries:
    passages = opensearch.search_body(query, numDocs=1000)["_id"].values
    for passage in passages:
        query_passage_pairs.append([query,passage])

query_passage_pairs

[["What is a physician's assistant?", 'MARCO_3951096'],
 ["What is a physician's assistant?", 'MARCO_849267'],
 ["What is a physician's assistant?", 'MARCO_920443'],
 ["What is a physician's assistant?", 'MARCO_2307794'],
 ["What is a physician's assistant?", 'MARCO_4389107'],
 ["What is a physician's assistant?", 'MARCO_8528286'],
 ["What is a physician's assistant?", 'MARCO_6607048'],
 ["What is a physician's assistant?", 'MARCO_4455128'],
 ["What is a physician's assistant?", 'MARCO_7447378'],
 ["What is a physician's assistant?", 'MARCO_7225587'],
 ["What is a physician's assistant?", 'MARCO_3757188'],
 ["What is a physician's assistant?", 'MARCO_7342284'],
 ["What is a physician's assistant?", 'MARCO_7514658'],
 ["What is a physician's assistant?", 'MARCO_2331422'],
 ["What is a physician's assistant?", 'MARCO_5891097'],
 ["What is a physician's assistant?",
  'CAR_027f0c602e6ed928e0eee643de549a83547775bb'],
 ["What is a physician's assistant?", 'MARCO_6607056'],
 ["What is a phys

In [31]:
# Extract CLS embeddings for all query-passage pairs
embeddings = np.zeros((len(query_passage_pairs), 768))  # Assuming 768 dimensions for BERT embeddings
embeddings = extract_cls(query_passage_pairs, embeddings=embeddings, batch_size=32)

In [32]:
# Get relevance scores from the classifier (regressor in case of continuous relevance)
scores = regressor.predict(embeddings)

In [36]:
# Sort passages by scores for each query
sorted_passages = {}

for i, query in enumerate(queries):
    # Get the corresponding query-passage pairs and scores
    query_scores = scores[i * len(passage): (i + 1) * len(passage)]  # Adjust based on your query-passages
    query_passages = query_passage_pairs[i * len(passage): (i + 1) * len(passage)]
    
    # Sort passages based on the classifier scores (descending order)
    sorted_passages[query] = sorted(zip(query_passages, query_scores), key=lambda x: x[1], reverse=True)

# Now `sorted_passages` is a dictionary with sorted passages for each query
# Print top 10 sorted passages for each query
for query, sorted_list in sorted_passages.items():
    print(f"Query: {query}")
    for passage, score in sorted_list[:5]:  # Print top 10 passages for each query
        print(f"Score: {score}, Passage: {passage}")

Query: What is a physician's assistant?
Score: 0.405670766342469, Passage: ["What is a physician's assistant?", 'MARCO_849267']
Score: 0.2147305739050417, Passage: ["What is a physician's assistant?", 'MARCO_3951096']


In [35]:

opensearch.get_doc_body("MARCO_4455128")

'1 The typical physician assistant is female. 2  The typical physician assistant graduated from PA school at around age 30. 3  The typical physician assistant has a bachelorâ\x80\x99s degree or more education. 4  The typical physician assistant is white. 5  The typical physician assistant has been out of school for 11 years, and is therefore about 41. What can I say â\x80\x93 on the whole, we are a middle aged profession. The is probably a reflection of the time it takes to accrue medically-related experience. The typical physician assistant practices in a group practice (29.5%)'